In [14]:
!pip install torch sentencepiece datasets tqdm

In [15]:
import torch
import torch.nn as nn
import math
from tqdm import tqdm
import sentencepiece as spm
from datasets import load_dataset

# Config
VOCAB_SIZE = 10000
D_MODEL = 256
N_LAYERS = 6
N_HEADS = 8
MAX_LEN = 512
BATCH_SIZE = 16
BLOCK_SIZE = 128
LR = 3e-4
STEPS = 5000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

text_data = [t for t in dataset["train"]["text"] if t.strip() != ""]
text = "\n".join(text_data)

print("Total characters:", len(text))

In [41]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

# TinyStories uses 'text' field directly
text_data = [t.strip() for t in dataset["train"]["text"] if len(t.strip()) > 20]

# Optional: limit size for faster training
text_data = text_data[:50000]   # adjust based on speed

text = "\n".join(text_data)

print("Total characters:", len(text))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Total characters: 44615770


In [42]:
with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [43]:
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='tokenizer',
    vocab_size=VOCAB_SIZE,
    model_type='bpe'
)

In [44]:
sp = spm.SentencePieceProcessor()
sp.load("tokenizer.model")

def encode(text):
    return sp.encode(text)

def decode(tokens):
    return sp.decode(tokens)

data = torch.tensor(encode(text), dtype=torch.long)
print("Total tokens:", len(data))

Total tokens: 10513700


In [45]:
def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [46]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = d_model // heads

        self.qkv = nn.Linear(d_model, d_model * 3)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        mask = torch.tril(torch.ones(T, T, device=x.device))
        mask = mask.unsqueeze(0).unsqueeze(0)

        scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().reshape(B, T, C)
        return self.fc(out)

In [47]:
class Block(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attn = SelfAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

In [48]:
class MiniLLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, D_MODEL)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)

        self.blocks = nn.Sequential(*[
            Block(D_MODEL, N_HEADS) for _ in range(N_LAYERS)
        ])

        self.ln = nn.LayerNorm(D_MODEL)
        self.fc = nn.Linear(D_MODEL, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        x = self.token_emb(x) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln(x)
        return self.fc(x)

In [49]:
model = MiniLLM(VOCAB_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

print("Model ready")

Model ready


In [53]:
import torch.nn.functional as F

for step in tqdm(range(STEPS)):
    xb, yb = get_batch(data, BLOCK_SIZE, BATCH_SIZE)

    logits = model(xb)
    loss = loss_fn(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 1. Cleaner logging
    if step % 500 == 0:
        print(f"\nStep {step} | Loss: {loss.item():.4f}")

    # 2. Sample generation to track quality
    if step % 2000 == 0:
        print("\nSample Output:")
        print(generate(model, "Machine learning is", temperature=0.7, top_k=30))
        print("\n" + "-"*50)

    # 3. Learning rate decay (important for refinement)
    if step == 15000:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-4
        print("\nLearning rate reduced to 1e-4\n")

  0%|          | 0/5000 [00:00<?, ?it/s]


Step 0 | Loss: 2.6479

Sample Output:


  0%|          | 4/5000 [00:00<08:09, 10.21it/s]

Machine learning is to be safe, so it is very nice to keep the special one for your journey. Once upon a time, there was a little girl. She was so happy that she said to one day, "I'm very lucky to be kind to

--------------------------------------------------


 10%|█         | 504/5000 [00:26<04:02, 18.52it/s]

...

 80%|████████  | 4005/5000 [03:25<01:12, 13.73it/s]

Machine learning is a fierce dragon who can help him. He can help us and give you a safe place. Come on, let's go get a rest." The dragon said, "O ⁇ , but be careful. I will permit us to come and get

--------------------------------------------------


 90%|█████████ | 4504/5000 [03:50<00:24, 20.20it/s]


Step 4500 | Loss: 2.5371


100%|██████████| 5000/5000 [04:16<00:00, 19.53it/s]


In [54]:
torch.save(model.state_dict(), "mini_llm_wikitext.pth")

In [55]:
import torch.nn.functional as F

def generate(model, start_text, max_new_tokens=50, temperature=0.8, top_k=50):
    model.eval()
    tokens = torch.tensor([encode(start_text)], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        logits = model(tokens[:, -BLOCK_SIZE:])
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, [-1]]] = -float('Inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return decode(tokens[0].tolist())

print(generate(model, "Machine learning is", temperature=0.7, top_k=30))

Machine learning is the answer. She says, "I'm sorry, little boy. I was mean. You are my new friend. I was rude. I didn't know what to do." The little boy was very disappointed. He said, "I
